In [1]:
import os
import random
import warnings
import time
import operator
from pathlib import Path
from PIL import Image
import cv2
import numpy as np

from joblib import Parallel, delayed, cpu_count
from deap import base, creator, tools, gp
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestRegressor as RF_CPU
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler

# ==========================================
# CẤU HÌNH HỆ THỐNG & THAM SỐ
# ==========================================
warnings.filterwarnings('ignore')
os.environ['PYTHONWARNINGS'] = 'ignore'

# Đặt seed để tái lập kết quả
random.seed(42)
np.random.seed(42)

# Tham số ES-FLGP
POP_SIZE = 100
MAX_GEN = 50
CXPB = 0.8
MUTPB = 0.19
ELITISM = int(0.01 * POP_SIZE)

# Tham số Surrogate
P_RATIO = 0.2  # Đánh giá thực tế top 20% cá thể
SURROGATE_SUBSET_SIZE = 0.1 # Dùng 10% tập train để lấy class_labels

# THƯ MỤC CHO WEBCAM
THU_MUC_GOC = '/kaggle/input/datasets/xixuhu/office31/Office-31/amazon'
THU_MUC_MOI = '/kaggle/working/amazon_split_50x50'
DATA_DIR_TRAIN = os.path.join(THU_MUC_MOI, 'train')
DATA_DIR_TEST = os.path.join(THU_MUC_MOI, 'test')

# ==========================================
# 1. HÀM TẢI DỮ LIỆU
# ==========================================
def prepare_train_test_dataset(src_dir, dest_dir, num_train=5, img_size=(50, 50)):
    duong_dan_goc = Path(src_dir)
    thu_muc_train = Path(dest_dir) / 'train'
    thu_muc_test = Path(dest_dir) / 'test'
    
    thu_muc_train.mkdir(parents=True, exist_ok=True)
    thu_muc_test.mkdir(parents=True, exist_ok=True)
    
    dinh_dang_anh = {'.jpg', '.jpeg', '.png', '.bmp'}
    tong_train = tong_test = 0
    
    print(f"Bắt đầu chia tập Train/Test (Train: {num_train} ảnh/class, Resize: {img_size})...", flush=True)
    
    for thu_muc_con in duong_dan_goc.iterdir():
        if not thu_muc_con.is_dir(): continue
        (thu_muc_train / thu_muc_con.name).mkdir(parents=True, exist_ok=True)
        (thu_muc_test / thu_muc_con.name).mkdir(parents=True, exist_ok=True)
        
        danh_sach_anh = [f for f in thu_muc_con.glob('*') if f.is_file() and f.suffix.lower() in dinh_dang_anh]
        random.shuffle(danh_sach_anh)
        
        so_luong_train = min(num_train, len(danh_sach_anh))
        anh_train = danh_sach_anh[:so_luong_train]
        anh_test = danh_sach_anh[so_luong_train:]
        
        def process_and_save(img_paths, dest_folder):
            count = 0
            for path in img_paths:
                try:
                    with Image.open(path) as img:
                        img = img.convert('RGB')
                        img_resized = img.resize(img_size, Image.Resampling.LANCZOS)
                        img_resized.save(dest_folder / path.name)
                        count += 1
                except Exception:
                    pass
            return count

        tong_train += process_and_save(anh_train, thu_muc_train / thu_muc_con.name)
        tong_test += process_and_save(anh_test, thu_muc_test / thu_muc_con.name)
        
    print(f"Hoàn thành! Tổng Train: {tong_train} ảnh | Tổng Test: {tong_test} ảnh\n" + "-"*50)

def load_dataset_raw(data_dir):
    images, labels = [], []
    if not os.path.exists(data_dir): return np.array([]), np.array([])
    class_names = sorted([d for d in os.listdir(data_dir) if os.path.isdir(os.path.join(data_dir, d))])
    class_mapping = {name: idx for idx, name in enumerate(class_names)}
    
    for class_name in class_names:
        class_dir = os.path.join(data_dir, class_name)
        for img_name in os.listdir(class_dir):
            img = cv2.imread(os.path.join(class_dir, img_name))
            if img is not None:
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                images.append((img / 255.0).astype(np.float32))
                labels.append(class_mapping[class_name])
    return np.array(images), np.array(labels, dtype=np.int32)

def get_stratified_subset(X, y, target_ratio=0.1):
    classes = np.unique(y)
    n_classes = len(classes)
    samples_per_class = max(1, int(len(X) * target_ratio / n_classes))
    selected_indices = []
    for cls in classes:
        cls_indices = np.where(y == cls)[0]
        selected = np.random.choice(cls_indices, min(samples_per_class, len(cls_indices)), replace=False)
        selected_indices.extend(selected)
    return X[np.array(selected_indices)], y[np.array(selected_indices)]

# ==========================================
# 2. TOÁN TỬ GP (OPENCV TỐI ƯU TỐC ĐỘ)
# ==========================================
def img_add(img1, img2): return np.clip(img1 + img2, 0.0, 1.0)
def img_sub(img1, img2): return np.clip(img1 - img2, 0.0, 1.0)
def img_max(img1, img2): return np.maximum(img1, img2)
def img_relu(img): return np.maximum(0, img)

def img_gaussian_blur(img):
    img_uint8 = np.clip(img * 255, 0, 255).astype(np.uint8)
    blurred = cv2.GaussianBlur(img_uint8, (5, 5), 1.0)
    return (blurred / 255.0).astype(np.float32)

def img_sobel_edges(img):
    img_uint8 = np.clip(img * 255, 0, 255).astype(np.uint8)
    sobelx = cv2.Sobel(img_uint8, cv2.CV_64F, 1, 0, ksize=3)
    sobely = cv2.Sobel(img_uint8, cv2.CV_64F, 0, 1, ksize=3)
    magnitude = cv2.magnitude(sobelx, sobely)
    magnitude = np.clip(magnitude / (np.max(magnitude) + 1e-5), 0, 1)
    return magnitude.astype(np.float32)

if "FitnessMax" not in creator.__dict__:
    creator.create("FitnessMax", base.Fitness, weights=(1.0,))
    creator.create("Individual", gp.PrimitiveTree, fitness=creator.FitnessMax, parent_fitness=list)

pset = gp.PrimitiveSet("MAIN", 1)
pset.renameArguments(ARG0="Image")
pset.addPrimitive(img_add, 2)
pset.addPrimitive(img_sub, 2)
pset.addPrimitive(img_max, 2)
pset.addPrimitive(img_gaussian_blur, 1)
pset.addPrimitive(img_sobel_edges, 1)
pset.addPrimitive(img_relu, 1)

toolbox = base.Toolbox()
toolbox.register("expr", gp.genHalfAndHalf, pset=pset, min_=1, max_=3)
toolbox.register("individual", tools.initIterate, creator.Individual, toolbox.expr)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)

# ==========================================
# 3. HÀM ĐÁNH GIÁ THỰC TẾ & BỘ NHỚ ĐỆM
# ==========================================
FITNESS_CACHE = {}

def gp_transform_data(individual, X_data, pset_local):
    func = gp.compile(expr=individual, pset=pset_local)
    num_features = 6 # Mean và Std
    X_features = np.zeros((X_data.shape[0], num_features))
    for i in range(X_data.shape[0]):
        try:
            processed_img = func(X_data[i])
            means = np.mean(processed_img, axis=(0, 1))
            stds = np.std(processed_img, axis=(0, 1))
            features = np.concatenate((means, stds))
            X_features[i, :] = np.nan_to_num(features, nan=0.0, posinf=1.0, neginf=0.0)
        except Exception:
            X_features[i, :] = 0.0 
    return X_features

def evaluate_singletree_core(ind, X_data, y_data, pset_local):
    tree_str = str(ind)
    if tree_str in FITNESS_CACHE:
        return FITNESS_CACHE[tree_str]

    X_features = gp_transform_data(ind, X_data, pset_local)
    if np.all(X_features == 0):
        FITNESS_CACHE[tree_str] = (0.0,)
        return (0.0,)
        
    clf = SVC(kernel='linear', max_iter=500, random_state=42)
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    
    accuracies = []
    for train_index, test_index in skf.split(X_features, y_data):
        X_train, X_test = X_features[train_index], X_features[test_index]
        y_train, y_test = y_data[train_index], y_data[test_index]
        
        scaler = StandardScaler()
        X_train = scaler.fit_transform(X_train)
        X_test = scaler.transform(X_test)
        
        try:
            clf.fit(X_train, y_train)
            accuracies.append(float(np.mean(clf.predict(X_test) == y_test)))
        except:
            accuracies.append(0.0)
            
    final_fitness = (np.mean(accuracies) * 100,)
    FITNESS_CACHE[tree_str] = final_fitness
    return final_fitness

def evaluate_wrapper(args):
    ind, X_data, y_data, pset_local = args
    return evaluate_singletree_core(ind, X_data, y_data, pset_local)

# ==========================================
# 4. ENSEMBLE SURROGATE MODULE
# ==========================================
def extract_surrogate_features(ind, X_sub, y_sub, pset_local):
    tree_feats = [len(ind), ind.height]
    parent_feats = getattr(ind, 'parent_fitness', [0.0, 0.0, 0.0])
    
    X_sub_features = gp_transform_data(ind, X_sub, pset_local)
    try:
        clf = SVC(kernel='linear', max_iter=500, random_state=42)
        clf.fit(X_sub_features, y_sub)
        preds = clf.predict(X_sub_features)
    except:
        preds = np.zeros(len(y_sub))
        
    return np.concatenate([tree_feats, parent_feats, preds])

def train_ensemble_surrogate(Archive, num_class_labels):
    X = np.array([item[0] for item in Archive])
    y = np.array([item[1] for item in Archive])
    
    idx_tree = 2
    idx_parent = idx_tree + 3
    
    rf_global = RF_CPU(n_estimators=100, max_depth=14, n_jobs=-1, random_state=42).fit(X, y)
    rf_tree = RF_CPU(n_estimators=100, max_depth=14, n_jobs=-1, random_state=42).fit(X[:, :idx_tree], y)
    rf_parent = RF_CPU(n_estimators=100, max_depth=14, n_jobs=-1, random_state=42).fit(X[:, idx_tree:idx_parent], y)
    rf_label = RF_CPU(n_estimators=100, max_depth=14, n_jobs=-1, random_state=42).fit(X[:, idx_parent:], y)
    
    return rf_global, rf_tree, rf_parent, rf_label

def predict_ensemble(surrogate_models, X_feats, w):
    rf_global, rf_tree, rf_parent, rf_label = surrogate_models
    idx_tree = 2
    idx_parent = idx_tree + 3
    
    y_g = rf_global.predict(X_feats)
    y_tree = rf_tree.predict(X_feats[:, :idx_tree])
    y_parent = rf_parent.predict(X_feats[:, idx_tree:idx_parent])
    y_label = rf_label.predict(X_feats[:, idx_parent:])
    
    eps = 1e-6
    y_g, y_tree = np.clip(y_g, eps, None), np.clip(y_tree, eps, None)
    y_parent, y_label = np.clip(y_parent, eps, None), np.clip(y_label, eps, None)
    
    y_e = w * np.log(y_g) + ((1.0 - w) / 3.0) * (np.log(y_tree) + np.log(y_parent) + np.log(y_label))
    return np.exp(y_e)

# ==========================================
# 5. CÁC TOÁN TỬ TIẾN HÓA LƯU PARENT FITNESS
# ==========================================
toolbox.register("select_tournament", tools.selTournament, tournsize=5)

def cx_wrapper(ind1, ind2):
    fit1 = ind1.fitness.values[0] if ind1.fitness.valid else 0.0
    fit2 = ind2.fitness.values[0] if ind2.fitness.valid else 0.0
    p_fit = [max(fit1, fit2), min(fit1, fit2), (fit1+fit2)/2.0]
    
    ind1, ind2 = gp.cxOnePoint(ind1, ind2)
    ind1.parent_fitness = p_fit
    ind2.parent_fitness = p_fit
    return ind1, ind2

def mut_wrapper(individual):
    fit = individual.fitness.values[0] if individual.fitness.valid else 0.0
    p_fit = [fit, fit, fit]
    
    individual, = gp.mutUniform(individual, expr=toolbox.expr, pset=pset)
    individual.parent_fitness = p_fit
    return individual,

toolbox.register("mate", cx_wrapper)
toolbox.register("mutate", mut_wrapper)

toolbox.decorate("mate", gp.staticLimit(key=operator.attrgetter("height"), max_value=8))
toolbox.decorate("mutate", gp.staticLimit(key=operator.attrgetter("height"), max_value=8))

# ==========================================
# 6. VÒNG LẶP TIẾN HÓA ES-FLGP (MAIN)
# ==========================================
def main(X_train_raw, y_train_raw, X_test_raw, y_test_raw):
    start_time_total = time.time()
    n_jobs = cpu_count()
    
    X_sub, y_sub = get_stratified_subset(X_train_raw, y_train_raw, target_ratio=SURROGATE_SUBSET_SIZE)
    
    print(f"\n[HỆ THỐNG] Đang chạy ES-FLGP (Đơn cây có Surrogate) - WEBCAM")
    print(f"[HỆ THỐNG] Đánh giá thực tế P_RATIO: {P_RATIO*100}% | 5-Fold CV")
    print(f"[HỆ THỐNG] Đa luồng: {n_jobs} cores | Tập con Surrogate: {len(X_sub)} ảnh\n")

    pop = toolbox.population(n=POP_SIZE)
    Archive = []  
    E_evals = 0
    E_max = POP_SIZE + (MAX_GEN * int(P_RATIO * POP_SIZE))

    # --- THẾ HỆ 0 ---
    print("Đang đánh giá thế hệ đầu tiên bằng hàm thực tế...")
    tasks = [(ind, X_train_raw, y_train_raw, pset) for ind in pop]
    results = Parallel(n_jobs=n_jobs, require='sharedmem')(delayed(evaluate_wrapper)(task) for task in tasks)
    
    for ind, res in zip(pop, results):
        ind.fitness.values = res     
        ind.parent_fitness = [res[0], res[0], res[0]]
        
        feats = extract_surrogate_features(ind, X_sub, y_sub, pset)
        Archive.append((feats, res[0]))
        E_evals += 1

    best_acc_global = max(res[0] for res in results)

    # --- VÒNG LẶP TIẾN HÓA ---
    for gen in range(MAX_GEN):
        start_time_gen = time.time()
        print(f"\n-- Thế hệ {gen+1}/{MAX_GEN} (E_evals: {E_evals}/{E_max}) --", flush=True)
        
        elites = tools.selBest(pop, k=ELITISM)
        elites = list(map(toolbox.clone, elites))
        
        num_select = POP_SIZE - ELITISM
        offspring = toolbox.select_tournament(pop, num_select)
        offspring = list(map(toolbox.clone, offspring))

        for child1, child2 in zip(offspring[::2], offspring[1::2]):
            if random.random() < CXPB:
                toolbox.mate(child1, child2)
                del child1.fitness.values
                del child2.fitness.values
                
        for mutant in offspring:
            if random.random() < MUTPB:
                toolbox.mutate(mutant)
                del mutant.fitness.values

        invalid_ind = [ind for ind in offspring if not ind.fitness.valid]
        to_predict = []
        
        # 1. Rà soát Cache
        for ind in invalid_ind:
            tree_str = str(ind)
            if tree_str in FITNESS_CACHE:
                ind.fitness.values = FITNESS_CACHE[tree_str]
            else:
                to_predict.append(ind)
                
        # 2. Áp dụng Ensemble Surrogate
        if to_predict:
            rf_models = train_ensemble_surrogate(Archive, len(X_sub))
            w = min(1.0, E_evals / E_max)
            
            X_pred_feats = np.array([extract_surrogate_features(ind, X_sub, y_sub, pset) for ind in to_predict])
            y_preds = predict_ensemble(rf_models, X_pred_feats, w)
            
            for ind, pred_fit in zip(to_predict, y_preds):
                ind.fitness.values = (pred_fit,)
            
            # 3. Chọn lọc TOP P_RATIO để đánh giá thực tế
            p_size = max(1, int(P_RATIO * POP_SIZE))
            to_predict.sort(key=lambda x: x.fitness.values[0], reverse=True)
            top_p_inds = to_predict[:p_size]
            
            # Đánh giá lại top P
            tasks_gen = [(ind, X_train_raw, y_train_raw, pset) for ind in top_p_inds]
            results_gen = Parallel(n_jobs=n_jobs, require='sharedmem')(delayed(evaluate_wrapper)(task) for task in tasks_gen)
            
            for ind, res, feats in zip(top_p_inds, results_gen, X_pred_feats[:p_size]):
                ind.fitness.values = res
                Archive.append((feats, res[0]))
                E_evals += 1
                
            # [LỖI LÙI LỀ ĐÃ ĐƯỢC KHẮC PHỤC]: Dọn dẹp Archive
            unique_archive = []
            seen = set()
            for feats, fit in Archive:
                feat_tuple = tuple(np.round(feats, decimals=4))
                if feat_tuple not in seen:
                    seen.add(feat_tuple)
                    unique_archive.append((feats, fit))
            Archive = unique_archive
            
            Archive.sort(key=lambda x: x[1], reverse=True)
            Archive = Archive[:POP_SIZE]

        pop[:] = elites + offspring
        
        best_ind_gen = tools.selBest(pop, 1)[0]
        if best_ind_gen.fitness.values[0] > best_acc_global:
            best_acc_global = best_ind_gen.fitness.values[0]
            
        print(f"   Độ chính xác tốt nhất thế : {best_ind_gen.fitness.values[0]:.2f}%")
        print(f"   Kỷ lục toàn cục           : {best_acc_global:.2f}%")
        print(f"   Thời gian chạy thế hệ     : {time.time() - start_time_gen:.2f} giây")

    # ==========================================
    # 7. ĐÁNH GIÁ TRÊN TẬP TEST ĐỘC LẬP
    # ==========================================
    print("\n================ ĐÁNH GIÁ TRÊN TẬP TEST ĐỘC LẬP ================")
    best_ind = tools.selBest(pop, 1)[0]
    
    X_train_feats = gp_transform_data(best_ind, X_train_raw, pset)
    X_test_feats = gp_transform_data(best_ind, X_test_raw, pset)
    
    scaler = StandardScaler()
    X_train_feats = scaler.fit_transform(X_train_feats)
    X_test_feats = scaler.transform(X_test_feats)
    
    clf = SVC(kernel='linear', max_iter=2000, random_state=42)
    clf.fit(X_train_feats, y_train_raw)
    test_acc = float(np.mean(clf.predict(X_test_feats) == y_test_raw)) * 100
    
    total_time = time.time() - start_time_total
    hours, rem = divmod(total_time, 3600)
    minutes, seconds = divmod(rem, 60)
    
    print("-" * 50)
    print(f"Độ chính xác Validation (Cao nhất) : {best_ind.fitness.values[0]:.2f}%")
    print(f"ĐỘ CHÍNH XÁC TEST CUỐI CÙNG        : {test_acc:.2f}%")
    print(f"TỔNG THỜI GIAN CHẠY ES-FLGP        : {int(hours)} giờ {int(minutes)} phút {seconds:.2f} giây")
    

if __name__ == "__main__":
    if not os.path.exists(THU_MUC_MOI):
        prepare_train_test_dataset(THU_MUC_GOC, THU_MUC_MOI, num_train=30, img_size=(50, 50))
    else:
        print(f"Thư mục {THU_MUC_MOI} đã tồn tại. Bỏ qua bước tiền xử lý.")

    print("\n--- ĐANG TẢI TẬP TRAIN ---")
    X_train, y_train = load_dataset_raw(DATA_DIR_TRAIN)
    print(f"Kích thước tập Train: {X_train.shape}")
    
    print("--- ĐANG TẢI TẬP TEST ---")
    X_test, y_test = load_dataset_raw(DATA_DIR_TEST)
    print(f"Kích thước tập Test: {X_test.shape}")

    if len(X_train) > 0 and len(X_test) > 0:
        main(X_train, y_train, X_test, y_test)
    else:
        print("Lỗi: Không tìm thấy dữ liệu! Vui lòng kiểm tra lại đường dẫn.")


Bắt đầu chia tập Train/Test (Train: 30 ảnh/class, Resize: (50, 50))...
Hoàn thành! Tổng Train: 930 ảnh | Tổng Test: 1887 ảnh
--------------------------------------------------

--- ĐANG TẢI TẬP TRAIN ---
Kích thước tập Train: (930, 50, 50, 3)
--- ĐANG TẢI TẬP TEST ---
Kích thước tập Test: (1887, 50, 50, 3)

[HỆ THỐNG] Đang chạy ES-FLGP (Đơn cây có Surrogate) - WEBCAM
[HỆ THỐNG] Đánh giá thực tế P_RATIO: 20.0% | 5-Fold CV
[HỆ THỐNG] Đa luồng: 4 cores | Tập con Surrogate: 93 ảnh

Đang đánh giá thế hệ đầu tiên bằng hàm thực tế...

-- Thế hệ 1/50 (E_evals: 100/1100) --
   Độ chính xác tốt nhất thế : 13.33%
   Kỷ lục toàn cục           : 13.33%
   Thời gian chạy thế hệ     : 15.06 giây

-- Thế hệ 2/50 (E_evals: 120/1100) --
   Độ chính xác tốt nhất thế : 13.33%
   Kỷ lục toàn cục           : 13.33%
   Thời gian chạy thế hệ     : 19.28 giây

-- Thế hệ 3/50 (E_evals: 140/1100) --
   Độ chính xác tốt nhất thế : 13.33%
   Kỷ lục toàn cục           : 13.33%
   Thời gian chạy thế hệ     : 20.45 g